# AWS Bedrock + LiteLLM Testing

Simple tests for Bedrock models via LiteLLM with cost tracking.

In [ ]:
# Imports
import os

import dspy
from datasets import load_dataset
from dotenv import load_dotenv
from dspy import GEPA, LM
from litellm import completion, completion_cost

load_dotenv()
print("✓ Loaded")

In [ ]:
# Config
region = os.getenv("AWS_REGION")
token = os.getenv("AWS_BEARER_TOKEN_BEDROCK")
llama_1b = os.getenv("BEDROCK_LLAMA_1B")
llama_3b = os.getenv("BEDROCK_LLAMA_3B")
llama_8b = os.getenv("BEDROCK_LLAMA_8B")
gpt_oss_20b = os.getenv("BEDROCK_GPT_OSS_20B")
gpt_oss_120b = os.getenv("BEDROCK_GPT_OSS_120B")

print(f"Region: {region}")
print(f"Token: {'✓' if token else '✗'}")
print(f"Llama Models: {llama_1b}, {llama_3b}, {llama_8b}")
print(f"GPT OSS Models: {gpt_oss_20b}, {gpt_oss_120b}")

## Test 1: Basic Call

In [ ]:
response = completion(
    model=f"bedrock/{llama_1b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=50,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

## Test 1b: GPT OSS 120B

In [ ]:
response = completion(
    model=f"bedrock/{gpt_oss_120b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=200,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

## Test 2: Math Reasoning

In [ ]:
response = completion(
    model=f"bedrock/{llama_8b}",
    messages=[
        {"role": "system", "content": "Think step by step"},
        {
            "role": "user",
            "content": "If I have 3 apples and buy 2 more, then give 1 away, how many?",
        },
    ],
    max_tokens=150,
    temperature=0,
)

print(response.choices[0].message.content)
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

## Test 3: Compare All Models

In [ ]:
question = "What is 2+2?"
total_cost = 0

for name, model in [("1B", llama_1b), ("3B", llama_3b), ("8B", llama_8b)]:
    response = completion(
        model=f"bedrock/{model}",
        messages=[{"role": "user", "content": question}],
        max_tokens=20,
        temperature=0,
    )

    cost = completion_cost(completion_response=response)
    total_cost += cost

    print(f"{name}: {response.choices[0].message.content}")
    print(f"     Tokens: {response.usage.total_tokens}, Cost: ${cost:.6f}\n")

print(f"Total: ${total_cost:.6f}")

## Test 5: Temperature Effects

In [ ]:
prompt = "The future of AI is"

for temp in [0.0, 0.7, 1.0]:
    response = completion(
        model=f"bedrock/{llama_3b}",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=temp,
    )

    print(f"Temp {temp}: {response.choices[0].message.content}\n")

## Test 6: DSPy with LiteLLM

In [ ]:
# Configure DSPy to use LiteLLM with Bedrock
lm = LM(model=f"bedrock/{llama_8b}", temperature=0, max_tokens=100)
dspy.configure(lm=lm)


# Define a simple signature
class BasicQA(dspy.Signature):
    """Answer questions concisely."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


# Create and test predictor
qa = dspy.ChainOfThought(BasicQA)
question = "What is the capital of France?"
response = qa(question=question)

print(f"Question: {question}")
print(f"Answer: {response.answer}")

# Inspect available fields
print(f"\nAvailable fields: {list(response.keys())}")

# Try to access reasoning if available
if hasattr(response, "reasoning"):
    print(f"\nReasoning: {response.reasoning}")
elif "reasoning" in response:
    print(f"\nReasoning: {response.reasoning}")

In [ ]:
# Access DSPy call history for cost tracking

# Get the last call from DSPy's history
last_call = lm.history[-1]

print("DSPy Call History:")
print(f"  Prompt tokens: {last_call['usage']['prompt_tokens']}")
print(f"  Completion tokens: {last_call['usage']['completion_tokens']}")
print(f"  Total tokens: {last_call['usage']['total_tokens']}")

# Calculate cost from the response
if "response" in last_call:
    cost = completion_cost(completion_response=last_call["response"])
    print(f"  Cost: ${cost:.6f}")
else:
    print("\n  Note: Raw response not available in history")
    print("  You can enable this with: lm = LM(..., cache=False)")

# GEPA/DSPy HotpotQA Task

Testing prompt optimization on multi-hop question answering.

In [ ]:
def init_dataset(num_samples=100):
    # Load HotpotQA dataset
    raw_data = load_dataset("hotpot_qa", "fullwiki")

    # Process train split
    raw_train = (
        raw_data["train"].shuffle(seed=42).select(range(min(len(raw_data["train"]), num_samples)))
    )
    train_split = [
        dspy.Example(
            {
                "question": x["question"],
                "answer": x["answer"],
            }
        ).with_inputs("question")
        for x in raw_train
    ]

    # Process validation split for test
    raw_val = (
        raw_data["validation"]
        .shuffle(seed=42)
        .select(range(min(len(raw_data["validation"]), num_samples)))
    )
    test_split = [
        dspy.Example(
            {
                "question": x["question"],
                "answer": x["answer"],
            }
        ).with_inputs("question")
        for x in raw_val
    ]

    # Split train into train/val
    tot_num = len(train_split)
    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num) :]
    test_set = test_split

    return train_set, val_set, test_set


train_set, val_set, test_set = init_dataset(num_samples=100)
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
print(f"Example: {train_set[0]}")

In [ ]:
# Configure DSPy to use LiteLLM with Bedrock (task agent)
lm = LM(model=f"bedrock/{gpt_oss_20b}", temperature=0.7, max_tokens=8192)
dspy.configure(lm=lm)

In [ ]:
class GenerateResponse(dspy.Signature):
    """Answer the question with a short, factual response."""

    question = dspy.InputField()
    answer = dspy.OutputField(desc="A short factual answer (1-5 words)")


program = dspy.ChainOfThought(GenerateResponse)


def normalize_answer(s):
    """Normalize answer for comparison."""
    if s is None:
        return ""
    return s.lower().strip()


def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    gold = normalize_answer(example["answer"])
    pred = normalize_answer(prediction.answer)
    if not pred:
        return 0
    # Exact match or gold contained in prediction
    return int(gold == pred or gold in pred or pred in gold)

In [ ]:
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True,
)

evaluate(program)

In [ ]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    gold = normalize_answer(example["answer"])
    pred = normalize_answer(prediction.answer)

    # Handle None/empty answer
    if not pred:
        feedback_text = (
            "Your response was empty or could not be parsed. "
            "Please provide a short, factual answer to the question."
        )
        feedback_text += f" The correct answer is '{example['answer']}'."
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Check for match
    score = int(gold == pred or gold in pred or pred in gold)

    if score == 1:
        feedback_text = f"Correct! The answer is '{example['answer']}'."
    else:
        feedback_text = f"Incorrect. You answered '{prediction.answer}', but the correct answer is '{example['answer']}'."
        feedback_text += " Make sure to provide a concise, factual answer."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [ ]:
optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=64,
    track_stats=True,
    reflection_minibatch_size=3,
    # Using gpt_oss_120b for reflection - larger model for instruction generation
    reflection_lm=dspy.LM(model=f"bedrock/{gpt_oss_120b}", temperature=1.0, max_tokens=4096),
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

In [ ]:
print(optimized_program.predict.signature.instructions)

In [ ]:
evaluate(optimized_program)

Generative AI was used to assist in building this notebook.